# Privugger Pyro Inference Methods

This notebook demonstrates the different inference methods available for the Pyro backend in Privugger:
- SVI only (Stochastic Variational Inference)
- MCMC only (No-U-Turn Sampler)
- Hybrid approach (SVI + MCMC)

It also shows how to use AutoGuide for automatic variational guide generation.

In [ ]:
import privugger as pv
import numpy as np
import matplotlib.pyplot as plt
import arviz as az

# Reset Privugger state
pv.reset()

## Example Model: BMI Calculator

We'll create a simple BMI (Body Mass Index) calculator as our example model.
BMI is calculated as weight (kg) divided by height (m) squared.

We'll specify priors for height and weight, and add a constraint that the BMI falls within a certain range.

In [ ]:
# Define priors
height = pv.Normal("height", mu=170.0, std=10.0)  # Height in cm
weight = pv.Normal("weight", mu=70.0, std=15.0)   # Weight in kg

# Define program for BMI calculation
def bmi_func(height, weight):
    # Convert height from cm to m
    height_m = height / 100.0
    # Calculate BMI
    return weight / (height_m * height_m)

# Create dataset and program
ds = pv.Dataset(input_specs=[height, weight])
prog = pv.Program("bmi", dataset=ds, output_type=pv.Float, function=bmi_func)

# Add observation that BMI is in the healthy range (between 18.5 and 24.9)
prog.add_observation("bmi >= 18.5 && bmi <= 24.9", precision=0.1)

## 1. SVI Only (Default Method)

First, let's run inference using SVI only. This is the default method for the Pyro backend.

In [ ]:
# Run inference with SVI method
svi_trace = pv.infer(
    prog, 
    method="pyro",             # Use Pyro backend
    pyro_method="svi",         # Use SVI only (default)
    svi_steps=500,             # Number of SVI steps
    draws=1000,                # Number of posterior samples
    chains=2,                  # Number of chains
    autoguide=False,           # Use manual guide
    target_idx=1               # Focus on weight (index 1)
)

# Plot the results
az.plot_posterior(svi_trace, var_names=["height", "weight"])
plt.tight_layout()
plt.show()

## 2. MCMC Only

Now, let's run inference using MCMC (NUTS) only, without SVI.

In [ ]:
# Run inference with MCMC method
mcmc_trace = pv.infer(
    prog, 
    method="pyro",             # Use Pyro backend
    pyro_method="mcmc",        # Use MCMC only
    draws=1000,                # Number of posterior samples
    chains=2,                  # Number of chains
    warmup_steps=200,          # Number of warmup steps
    target_idx=1               # Focus on weight (index 1)
)

# Plot the results
az.plot_posterior(mcmc_trace, var_names=["height", "weight"])
plt.tight_layout()
plt.show()

## 3. Hybrid Approach (SVI + MCMC)

Now, let's run inference using the hybrid approach, which combines SVI and MCMC:

In [ ]:
# Run inference with hybrid method
hybrid_trace = pv.infer(
    prog, 
    method="pyro",             # Use Pyro backend
    pyro_method="hybrid",      # Use hybrid SVI+MCMC
    svi_steps=300,             # Number of SVI steps
    draws=1000,                # Number of posterior samples
    chains=2,                  # Number of chains
    warmup_steps=200,          # Number of warmup steps
    target_idx=1               # Focus on weight (index 1)
)

# Plot the results
az.plot_posterior(hybrid_trace, var_names=["height", "weight"])
plt.tight_layout()
plt.show()

## 4. Using AutoGuide

Finally, let's run inference using SVI with AutoGuide instead of the manual guide:

In [ ]:
# Run inference with SVI method and AutoGuide
autoguide_trace = pv.infer(
    prog, 
    method="pyro",             # Use Pyro backend
    pyro_method="svi",         # Use SVI
    svi_steps=500,             # Number of SVI steps
    draws=1000,                # Number of posterior samples
    chains=2,                  # Number of chains
    autoguide=True,            # Use AutoGuide
    target_idx=1               # Focus on weight (index 1)
)

# Plot the results
az.plot_posterior(autoguide_trace, var_names=["height", "weight"])
plt.tight_layout()
plt.show()

## Comparing Results

Let's compare the results from the different inference methods:

In [ ]:
# Combine the traces into a single dataset for comparison
combined_trace = az.concat(
    dict(
        svi=svi_trace,
        mcmc=mcmc_trace,
        hybrid=hybrid_trace,
        autoguide=autoguide_trace
    ),
    dim="chain"
)

# Plot density comparison
az.plot_density(
    combined_trace,
    var_names=["height", "weight"],
    shade=0.5,
    hdi_prob=0.95
)
plt.tight_layout()
plt.show()

## Constraint Satisfaction Check

Let's check how well each method satisfies the BMI constraint:

In [ ]:
def check_bmi_constraint(trace):
    # Get posterior samples
    height_samples = trace.posterior["height"].values.flatten()
    weight_samples = trace.posterior["weight"].values.flatten()
    
    # Calculate BMI for all samples
    height_m = height_samples / 100.0  # Convert to meters
    bmi = weight_samples / (height_m * height_m)
    
    # Check constraints
    valid = (bmi >= 18.5) & (bmi <= 24.9)
    
    # Calculate statistics
    percent_valid = np.mean(valid) * 100
    mean_bmi = np.mean(bmi)
    std_bmi = np.std(bmi)
    
    return {
        "percent_valid": percent_valid,
        "mean_bmi": mean_bmi,
        "std_bmi": std_bmi
    }

# Check constraint satisfaction for each method
methods = ["SVI", "MCMC", "Hybrid", "AutoGuide"]
traces = [svi_trace, mcmc_trace, hybrid_trace, autoguide_trace]

results = []
for method, trace in zip(methods, traces):
    stats = check_bmi_constraint(trace)
    results.append((method, stats))
    print(f"{method}:")
    print(f"  Valid BMI percentage: {stats['percent_valid']:.2f}%")
    print(f"  Mean BMI: {stats['mean_bmi']:.2f}")
    print(f"  Std BMI: {stats['std_bmi']:.2f}")
    print()

## Summary

In this notebook, we've demonstrated the different inference methods available for the Pyro backend in Privugger:

1. **SVI (only)**: Fast approximate inference, ideal for initial exploration and parameter sweeps.
2. **MCMC (only)**: Slower but more accurate MCMC sampling without any variational approximation.
3. **Hybrid (SVI+MCMC)**: Combines the speed of SVI for initialization with the accuracy of MCMC.
4. **AutoGuide**: Automatic guide generation for SVI, useful when creating custom guides is difficult.

Based on the constraint satisfaction results, we can observe which method best enforces the BMI constraints in our model.

### When to use each method:

* **SVI**: Use for rapid "what-if" exploration and parameter sweeps.
* **Hybrid**: Use for final leakage numbers for a compliance report where accuracy is important but efficiency is also needed.
* **MCMC**: Use when you distrust variational initialization or have tiny models that don't need the speed advantage of SVI.
* **AutoGuide**: Use when the default guide doesn't perform well or for complex models where designing a custom guide would be challenging.